In [ ]:
import json
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

REPO_ROOT = Path().resolve().parent
REPOS_FILE = REPO_ROOT / "data" / "repos.json"
SCRIPTS_DIR = REPO_ROOT / "scripts"
RESULTS_DIR = REPO_ROOT / "results"
REPORTS_DIR = REPO_ROOT / "evidence" / "reportes"
CAPTURAS_DIR = REPO_ROOT / "evidence" / "capturas"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
CAPTURAS_DIR.mkdir(parents=True, exist_ok=True)

print("Verificando config de repos...")
try:
    with open(REPOS_FILE, "r", encoding="utf-8") as f:
        repos_data = json.load(f)
        repos_list = repos_data.get("repositories", [])

    if len(repos_list) == 0:
        print("Error: No hay repositorios configurados en repos.json")
        raise Exception(
            "Pipeline stopped: Configura repositorios antes de continuar")
    else:
        print(f"Se encontraron {len(repos_list)} repositorios para analizar\n")

except Exception as e:
    print(f"Error al leer repos.json: {e}")
    raise


def ejecutar_script(descripcion, script_name):
    script_path = SCRIPTS_DIR / script_name
    print(f"\n▶ {descripcion}")

    resultado = subprocess.run(
        [sys.executable, str(script_path)],
        cwd=REPO_ROOT,
        capture_output=True,
        text=True
    )

    if resultado.returncode == 0:
        print(f"{script_name} finalizó correctamente")
    else:
        print(f"Advertencia/Error en {script_name}.")
        print(f"  Detalle: {resultado.stderr.strip()[:200]}...")
        print("  (Revisar en /evidence/logs/ para ver el detalle completo)")

In [ ]:
ejecutar_script("Clonando/Actualizando repositorios...", "add_submodules.py")

In [ ]:
ejecutar_script("Generando SBOMs...", "generate_sboms.py")

In [ ]:
ejecutar_script(
    "Escaneando vulnerabilidades en dependencias (Grype)...", "generate_grype.py")

In [ ]:
ejecutar_script(
    "Escaneando vulnerabilidades en código fuente (CodeQL)...", "generate_codeql.py")

In [ ]:
print(f"Buscando resultados en: {RESULTS_DIR}")
grype_data = []

for filepath in RESULTS_DIR.glob("*-grype.json"):
    repo_name = filepath.name.replace("-grype.json", "")

    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
        for vuln in data.get("vulnerabilities", []):
            grype_data.append({
                "Repositorio": repo_name,
                "Vector de Ataque": "Dependencias (SCA)",
                "Vulnerabilidad": vuln.get("vuln_id"),
                "Severidad": vuln.get("vuln_severity").capitalize(),
                "Componente Afectado": f"{vuln.get('package_name')} @ {vuln.get('current_version')}",
                "Solución Propuesta": vuln.get("fix_version", "N/A"),
                "Descripción": vuln.get("message")
            })

df_grype = pd.DataFrame(grype_data)
print(f"Se encontraron {len(df_grype)} vulnerabilidades en dependencias.")

In [ ]:
codeql_data = []

for filepath in RESULTS_DIR.glob("*-codeql.json"):
    repo_name = filepath.name.replace("-codeql.json", "")

    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
        for issue in data.get("issues", []):
            level = issue.get("level", "note")
            severity_map = {"error": "High",
                            "warning": "Medium", "note": "Low"}

            codeql_data.append({
                "Repositorio": repo_name,
                "Vector de Ataque": "Código Fuente (SAST)",
                "Vulnerabilidad": issue.get("rule_id"),
                "Severidad": severity_map.get(level, "Unknown"),
                "Componente Afectado": issue.get("file"),
                "Solución Propuesta": "Requiere parche o revisión manual",
                "Descripción": issue.get("message")
            })

df_codeql = pd.DataFrame(codeql_data)
print(
    f"Se encontraron {len(df_codeql)} problemas de seguridad en código fuente.")

In [ ]:
df_consolidado = pd.concat([df_grype, df_codeql], ignore_index=True)

if not df_consolidado.empty:
    orden_severidad = {"Critical": 1, "High": 2,
                       "Medium": 3, "Low": 4, "Unknown": 5}
    df_consolidado["Prioridad_Num"] = df_consolidado["Severidad"].map(
        orden_severidad)

    df_consolidado = df_consolidado.sort_values(
        by=["Prioridad_Num", "Repositorio"]).drop(columns=["Prioridad_Num"])

    print("TOP 10 Vulnerabilidades Más Críticas (Prioridad de Mitigación):")
    display(df_consolidado.head(10))

    ruta_csv = REPORTS_DIR / "resumen_vulnerabilidades.csv"
    df_consolidado.to_csv(ruta_csv, index=False, encoding="utf-8")
    print(f"\nReporte exportado a: {ruta_csv.relative_to(REPO_ROOT)}")
else:
    print("No hay datos para procesar.")

In [ ]:
if not df_consolidado.empty:
    plt.figure(figsize=(10, 6))

    conteos = df_consolidado['Severidad'].value_counts()

    colores = []
    for sev in conteos.index:
        if sev == 'Critical':
            colores.append('#8B0000')
        elif sev == 'High':
            colores.append('#FF4500')
        elif sev == 'Medium':
            colores.append('#FFD700')
        else:
            colores.append('#32CD32')

    bars = conteos.plot(kind='bar', color=colores, edgecolor='black')

    plt.title('Distribución de Vulnerabilidades por Severidad',
              fontsize=14, pad=15)
    plt.xlabel('Nivel de Severidad', fontsize=12)
    plt.ylabel('Cantidad de Hallazgos', fontsize=12)
    plt.xticks(rotation=0)
    plt.grid(axis='y', linestyle='--', alpha=0.7)

    ruta_grafico = CAPTURAS_DIR / "distribucion_severidad.png"
    plt.savefig(ruta_grafico, bbox_inches='tight', dpi=300)
    print(f"Gráfico guardado en: {ruta_grafico.relative_to(REPO_ROOT)}")

    plt.show()
else:
    print("No se generó gráfico porque no hay datos consolidados.")